In [1]:
pip install transformers dataset soundfile torch gradio

In [ ]:
import torch
import soundfile as sf
import gradio as gr
from transformers import pipeline,SpeechT5Processor,SpeechT5ForTextToSpeech,SpeechT5HifiGan

In [ ]:
# Load Models

# STT model (Whisper small)
stt_pipeline = pipeline("automatic-speech-recognition", model="openai/whisper-small")

# TTS model (SpeechT5 + vocoder)
processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
tts_model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

# Dummy speaker embedding
speaker_embeddings = torch.randn(1,512)


In [ ]:
def speech_to_text(audio):
    """Convert speech to text"""
    text = stt_pipeline(audio)["text"]
    return text

def text_to_speech(text):
    """Convert text to speech"""
    inputs = processor(text=text, return_tensors="pt")
    speech = tts_model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)
    sf.write("output.wav", speech.numpy(), 16000)
    return "output.wav"


In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("Speech-to-Text & Text-to-Speech Demo")

    with gr.Tab("Speech to Text"):
        audio_in = gr.Audio(type="filepath", label="Record or Upload Audio")
        text_out = gr.Textbox(label="Transcribed Text")
        stt_btn = gr.Button("Transcribe")
        stt_btn.click(speech_to_text, inputs=audio_in, outputs=text_out)

    with gr.Tab("Text to Speech"):
        text_in = gr.Textbox(label="Enter Text")
        audio_out = gr.Audio(label="Generated Speech")
        tts_btn = gr.Button("Generate Speech")
        tts_btn.click(text_to_speech, inputs=text_in, outputs=audio_out)

demo.launch()

